# 7. Pandas CSV Batch Sentiment

**Practical use case:** Analyze 20 reviews and add sentiment, confidence, topic and action columns.

This trainer-ready notebook contains explanation, live `langchain_openai` code, validation guidance and exercises. It makes real API calls and may incur charges.

## 1. Business problem

**Scenario:** Read 20 customer reviews, call the LLM for each row and add sentiment columns.

The goal is to convert an unstructured language task into a repeatable workflow that can be demonstrated, tested and later integrated into an application.

## 2. Solution workflow

1. Prepare or load the input.
2. Define the model and important parameters.
3. Construct a precise prompt or schema.
4. Invoke the model.
5. inspect and validate the response.
6. Save or pass the result to the next application step.

### 1. Set up the API key and model

This cell imports the required classes, reads the API key securely, and selects the model without exposing credentials.

**Expected result:** No model output is produced; the environment becomes ready for later API calls. Read the output before continuing to the next cell.

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

### 2. Load and inspect the CSV data

This cell loads the supplied CSV into a Pandas DataFrame and previews its shape and records before analysis.

**Expected result:** A table appears with the source rows and the expected five input columns. Read the output before continuing to the next cell.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/customer_reviews_20.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/customer_reviews_20.csv")
df = pd.read_csv(DATA_PATH)
print(df.shape)
display(df.head())

### 3. Define and validate structured output

This cell defines a Pydantic schema and configures structured output so model results have predictable fields and types.

**Expected result:** A validated Python object or dictionary is returned instead of unstructured text. Read the output before continuing to the next cell.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class ReviewAnalysis(BaseModel):
    sentiment: Literal["Positive", "Negative", "Neutral"]
    confidence: float = Field(ge=0, le=1)
    topic: str
    recommended_action: str

analyzer = ChatOpenAI(model=MODEL_NAME, temperature=0).with_structured_output(ReviewAnalysis)

def analyze_review(text):
    return analyzer.invoke(f"Analyze this customer review. Keep the action short: {text}")

results = [analyze_review(text) for text in df["review_text"]]
df["sentiment"] = [r.sentiment for r in results]
df["confidence"] = [r.confidence for r in results]
df["topic"] = [r.topic for r in results]
df["recommended_action"] = [r.recommended_action for r in results]
display(df)

### 4. Save the enriched CSV results

This cell saves the enriched results as a new CSV so they can be reused in reports or downstream applications.

**Expected result:** The output path and a small summary of the saved analysis are displayed. Read the output before continuing to the next cell.

In [ ]:
OUTPUT_PATH = DATA_PATH.parent / "customer_reviews_with_sentiment.csv"
df.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)
print(df["sentiment"].value_counts())

## Expected result

The model should return an output that follows the requested scope and format. During training, compare the actual response with the requirement instead of assuming that a fluent response is correct.

## Parameter experiment

Run the example once with `temperature=0` and once with a higher supported temperature. Compare consistency, wording and creativity. Keep other inputs unchanged so the comparison is meaningful.

## Output validation

Check that the response:

- follows the requested format;
- contains no invented facts;
- preserves important names and numbers;
- is relevant to the business question;
- can be safely used by the next system.